# Estado de enriquecimiento de los CVE en el NVD, por año de publicación

Cuaderno de lectura de la medición `nvd-fichas/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Respalda el artículo [treinta-y-tres-mil-sin-ficha](https://manpla.net/posts/treinta-y-tres-mil-sin-ficha/). Carga el fichero de al lado —o lo descarga del repositorio si se ejecuta fuera de él—, muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no vuelve a tomar la instantánea: para eso está `generar.py`.

*Reading notebook for this measurement: loads the file next to it, prints the provenance record and draws one figure. Column names are in Spanish; `GLOSARIO.md` gives the English form.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/nvd-fichas/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")


## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
a = leer("por-anio.csv")
print("filas por CVE:", len(leer("cve-estados.csv.gz")))
a[["anio", "publicados", "ficha_completa", "aplazados", "recibidos", "en_espera", "en_analisis", "explotados", "pct_ficha_completa", "pct_aplazados"]]

## Una figura

In [ ]:
m = leer("por-mes.csv")
fig, (x, y) = plt.subplots(1, 2, figsize=(12, 4.2))
x.bar(a.anio.astype(str), a.pct_ficha_completa, color="C0", label="ficha completa"); x.bar(a.anio.astype(str), a.pct_aplazados, bottom=a.pct_ficha_completa, color="C3", label="aplazados")
x.set_ylabel("% de los CVE publicados ese año"); x.set_title("con ficha completa y aplazados, por año"); x.legend(loc="lower left")
y.bar(m.mes, m.pct_aplazados, color="C3"); y.set_ylabel("% aplazados"); y.set_title("aplazados por mes de publicación, " + str(a.anio.iloc[-1])); y.tick_params(axis="x", rotation=45); plt.tight_layout()